# Adjective Analysis Across Languages: Pokémon Gamescript in Translation

This notebook compares adjective usage across four versions of a Pokémon gamescript:
- **English** (source text)
- **German** (source text)
- **Arabic translated from English** (via Google Translate)
- **Arabic translated from German** (via Google Translate)

Our goal is to extract adjectives from each text using part-of-speech (POS) tagging and compare them across languages. For English and German, we'll use **spaCy**. For Arabic, we'll use **CAMeL Tools**, a library purpose-built for Arabic NLP.

> **Why not spaCy for Arabic?** spaCy does have an Arabic model (`ar_core_news_sm`), but as of spaCy v3.8 it is no longer available as a compatible download — running `python -m spacy download ar_core_news_sm` returns an error. This turns out to be fine for our purposes: CAMeL Tools was purpose-built for Arabic morphological analysis and is considered more accurate than spaCy's Arabic model was, particularly on machine-translated text. So we're using the better tool by necessity!

### Installs
Before running this notebook, set up and activate your virtual environment (`.venv/`) in the project's `python/` folder,
and do the following installations in that activated virtual environment:
(Note: the first one installs the notebook library you need for running this Jupyter notebook.)

```
pip install notebook 
pip install spacy
pip install camel-tools
```

### Language Model Downloads
You'll also need to download spaCy's language models and CAMeL Tools' data. Run these **once** in your shell and make sure they install completely (will take some time). 

```
python -m spacy download en_core_web_lg
python -m spacy download de_core_news_lg
camel_data -i light
```

### Your Text Files
This notebook expects four text files. Update the file paths in the **Configuration** cell below to match where yours are stored in relation to this file:
- `EN_FILE` — English gamescript
- `DE_FILE` — German gamescript  
- `AR_FROM_EN_FILE` — Arabic translated from English
- `AR_FROM_DE_FILE` — Arabic translated from German

## Part 1: Setup and Configuration

In [ ]:
# Smoke test: Check that this notebook is running from your activated .venv in the project GitHub repo! 
import sys
print(sys.executable)

In [ ]:
# -------------------------------------------------------------------
# CONFIGURATION: Update these file paths to match your directory layout
# -------------------------------------------------------------------

EN_FILE      = '../pokecorpus-bw-scripts/en_script-REGEX.txt'
DE_FILE      = '../pokecorpus-bw-scripts/de_script-REGEX.txt'
AR_FROM_EN_FILE = '../arabic-translations/en-script-REGEX-to-ar.txt'
AR_FROM_DE_FILE = '../arabic-translations/de-script-REGEX-to-ar.txt'

# Output file paths
TSV_OUTPUT   = 'output/adjective_comparison.tsv'
DETAIL_OUTPUT = 'output/adjective_detail.tsv'

In [ ]:
# Imports — run this cell before anything else
import spacy
from collections import Counter
from IPython.display import HTML, display
import os

# Make sure the output directory exists
os.makedirs('output', exist_ok=True)

print("Imports loaded successfully.")

In [ ]:
# -------------------------------------------------------------------
# Load spaCy models
# Comment out the download lines after your first run!
# -------------------------------------------------------------------

# spacy.cli.download("en_core_web_lg")
# spacy.cli.download("de_core_news_lg")

nlp_en = spacy.load("en_core_web_lg")
nlp_de = spacy.load("de_core_news_lg")

print("spaCy models loaded: English (lg), German (lg)")

In [ ]:
# -------------------------------------------------------------------
# Load CAMeL Tools disambiguator for Arabic
# -------------------------------------------------------------------
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.disambig.mle import MLEDisambiguator

# This loads the pretrained morphological disambiguator for Modern Standard Arabic (MSA)
# It may take a moment the first time
mle = MLEDisambiguator.pretrained()

print("CAMeL Tools disambiguator loaded.")

## Part 2: Helper Functions

Here we define reusable functions for reading files and extracting adjectives. 
We write them once so we don't repeat the same code four times — once per language.

In [ ]:
def read_text(filepath, encoding='utf-8'):
    """Open and read a text file. Returns the full text as a string."""
    with open(filepath, 'r', encoding=encoding) as f:
        return f.read()


def get_adjectives_spacy(text, nlp):
    """
    Use a spaCy model to extract adjectives from text.
    Returns a list of (token_text, lemma) tuples for all ADJ tokens.
    
    spaCy uses Universal Dependencies (UD) POS tags, so ADJ is consistent
    across the English and German models.
    """
    doc = nlp(text)
    return [(token.text, token.lemma_) for token in doc if token.pos_ == "ADJ"]


def get_adjectives_camel(text):
    """
    Use CAMeL Tools to extract adjectives from Arabic text.
    Returns a list of (token_text, lemma) tuples for tokens tagged as 'adj'.
    
    CAMeL uses its own tagset (closer to traditional Arabic grammar).
    The POS tag for adjective in CAMeL is 'adj' (lowercase).
    
    Note: CAMeL returns a ranked list of possible analyses per token
    (Arabic is morphologically ambiguous). We take analyses[0], the top-ranked one.
    """
    tokens = simple_word_tokenize(text)
    analyses = mle.disambiguate(tokens)
    results = []
    for token, analysis in zip(tokens, analyses):
        if analysis.analyses:  # guard against tokens with no analysis
            top = analysis.analyses[0].analysis
            pos = top.get('pos', 'unknown')
            lemma = top.get('lex', token)  # 'lex' is CAMeL's lemma field
            if pos == 'adj':
                results.append((token, lemma))
    return results


def adjective_frequency(adj_list):
    """
    Given a list of (token, lemma) tuples, count frequency of each lemma.
    Returns a Counter of lemma -> count, sorted most common first.
    """
    lemmas = [lemma for _, lemma in adj_list]
    return Counter(lemmas)

def print_arabic(text, label=None):
    """
    Display Arabic text with correct right-to-left (RTL) rendering in Jupyter.
    Standard print() does not reliably render RTL text in notebook output cells.
    This function uses HTML + CSS to force correct directionality!
    
    label: optional heading string to display above the text.
    """
    heading = f'<p><strong>{label}</strong></p>' if label else ''
    display(HTML(
        f'{heading}<p dir="rtl" style="text-align:right; font-size:15px; '
        f'font-family: Arial, sans-serif; line-height: 1.8;">'
        f'{text}</p>'
    ))

print("Helper functions defined.")

## Part 3: Read the Texts

Let's open all four files and do a quick sanity check — printing the first 300 characters
of each so we can confirm the files loaded correctly and the encoding looks right.

In [ ]:
text_en      = read_text(EN_FILE)
text_de      = read_text(DE_FILE)
text_ar_en   = read_text(AR_FROM_EN_FILE)
text_ar_de   = read_text(AR_FROM_DE_FILE)

print("=== English (first 300 chars) ===")
print(text_en[:300])

print("\n=== German (first 300 chars) ===")
print(text_de[:300])

# For Arabic, we use print_arabic() so Jupyter renders RTL correctly!
print_arabic(text_ar_en[:300], label="Arabic from English (first 300 chars)")
print_arabic(text_ar_de[:300], label="Arabic from German (first 300 chars)")

## Part 4: Full Adjective Extraction

Now we run the adjective extraction on the **full texts**.
For English and German we use spaCy; for Arabic we use CAMeL Tools.

In [ ]:
# --- Extract adjectives from all four texts ---
# This may take a minute for large files — spaCy processes text in full pipeline

print("Processing English...")
adjs_en = get_adjectives_spacy(text_en, nlp_en)
print(f"  Found {len(adjs_en)} adjective tokens in English.")

print("Processing German...")
adjs_de = get_adjectives_spacy(text_de, nlp_de)
print(f"  Found {len(adjs_de)} adjective tokens in German.")

print("Processing Arabic (from English) with CAMeL Tools...")
adjs_ar_en = get_adjectives_camel(text_ar_en)
print(f"  Found {len(adjs_ar_en)} adjective tokens in Arabic-from-English.")

print("Processing Arabic (from German) with CAMeL Tools...")
adjs_ar_de = get_adjectives_camel(text_ar_de)
print(f"  Found {len(adjs_ar_de)} adjective tokens in Arabic-from-German.")

print("\nDone!")

## Part 5: Frequency Counts and Basic Statistics

Let's count how often each adjective lemma appears in each text,
and look at some basic statistics before we write to file.

In [ ]:
# Build frequency counters for each language
freq_en    = adjective_frequency(adjs_en)
freq_de    = adjective_frequency(adjs_de)
freq_ar_en = adjective_frequency(adjs_ar_en)
freq_ar_de = adjective_frequency(adjs_ar_de)

# Quick statistics summary
def token_count(text):
    """Rough word/token count — splits on whitespace."""
    return len(text.split())

totals = {
    'English':            (len(adjs_en),    token_count(text_en)),
    'German':             (len(adjs_de),    token_count(text_de)),
    'Arabic (from EN)':   (len(adjs_ar_en), token_count(text_ar_en)),
    'Arabic (from DE)':   (len(adjs_ar_de), token_count(text_ar_de)),
}

print("=== Adjective Statistics ===")
print("{:<22} {:>12} {:>12} {:>14}".format(
    "Text", "Total tokens", "Adj tokens", "Adj density %"))
print("-" * 62)
for lang, (adj_count, tok_count) in totals.items():
    density = (adj_count / tok_count * 100) if tok_count > 0 else 0
    print("{:<22} {:>12,} {:>12,} {:>13.2f}%".format(
        lang, tok_count, adj_count, density))

In [ ]:
# Top 20 most frequent adjective lemmas in each language
TOP_N = 20

for label, freq in [("English", freq_en), 
                    ("German", freq_de),
                    ("Arabic (from EN)", freq_ar_en),
                    ("Arabic (from DE)", freq_ar_de)]:
    print(f"\n=== Top {TOP_N} adjectives: {label} ===")
    print("{:<5} {:<30} {:>8}".format("Rank", "Lemma", "Count"))
    print("-" * 45)
    for rank, (lemma, count) in enumerate(freq.most_common(TOP_N), start=1):
        print("{:<5} {:<30} {:>8}".format(rank, lemma, count))

## Part 6: Write Output to TSV Files

We'll write two output files:

1. **`adjective_comparison.tsv`** — side-by-side top adjectives per language (good for a quick overview)
2. **`adjective_detail.tsv`** — one row per adjective token, with language, token, and lemma (good for deeper analysis)

In [ ]:
# --- Output File 1: Top adjectives side-by-side ---
# Each column = one language; rows = rank 1 through TOP_N

TOP_N_OUT = 50  # How many top adjectives to include in the comparison table

top_en    = freq_en.most_common(TOP_N_OUT)
top_de    = freq_de.most_common(TOP_N_OUT)
top_ar_en = freq_ar_en.most_common(TOP_N_OUT)
top_ar_de = freq_ar_de.most_common(TOP_N_OUT)

# Pad shorter lists so all columns are the same length
max_len = max(len(top_en), len(top_de), len(top_ar_en), len(top_ar_de))
def pad(lst, length):
    return lst + [('', '')] * (length - len(lst))

top_en    = pad(top_en, max_len)
top_de    = pad(top_de, max_len)
top_ar_en = pad(top_ar_en, max_len)
top_ar_de = pad(top_ar_de, max_len)

with open(TSV_OUTPUT, 'w', encoding='utf-8') as out:
    # Header
    out.write('\t'.join([
        'rank',
        'EN_lemma', 'EN_count',
        'DE_lemma', 'DE_count',
        'AR_from_EN_lemma', 'AR_from_EN_count',
        'AR_from_DE_lemma', 'AR_from_DE_count'
    ]) + '\n')
    
    for i in range(max_len):
        row = [
            str(i + 1),
            top_en[i][0],    str(top_en[i][1]),
            top_de[i][0],    str(top_de[i][1]),
            top_ar_en[i][0], str(top_ar_en[i][1]),
            top_ar_de[i][0], str(top_ar_de[i][1]),
        ]
        out.write('\t'.join(row) + '\n')

print(f"Wrote comparison table to: {TSV_OUTPUT}")

In [ ]:
# --- Output File 2: Full adjective detail, one row per token ---
# Columns: language, tagger, token (as it appears in text), lemma

all_data = [
    ('English',          'spaCy-en_core_web_lg',       adjs_en),
    ('German',           'spaCy-de_core_news_lg',       adjs_de),
    ('Arabic_from_EN',   'CAMeL-MLEDisambiguator',      adjs_ar_en),
    ('Arabic_from_DE',   'CAMeL-MLEDisambiguator',      adjs_ar_de),
]

with open(DETAIL_OUTPUT, 'w', encoding='utf-8') as out:
    out.write('\t'.join(['language', 'tagger', 'token', 'lemma']) + '\n')
    for language, tagger, adj_list in all_data:
        for token, lemma in adj_list:
            out.write('\t'.join([language, tagger, token, lemma]) + '\n')

print(f"Wrote detail file to: {DETAIL_OUTPUT}")
print(f"Total rows written: {sum(len(a) for _, _, a in all_data):,}")

## Next Steps

You now have:
- **`adjective_comparison.tsv`** — top adjectives ranked by frequency, one column per language
- **`adjective_detail.tsv`** — every adjective token in all four texts, with language and lemma info

Some questions to explore with these outputs:

1. **Adjective density**: Which translation is most "adjective-heavy"? Does Arabic gain or lose adjectives relative to English and German?
2. **Top adjectives**: Do the most common adjectives in the Arabic translations match what you'd expect from the English and German sources?
3. **Arabic-from-English vs. Arabic-from-German**: Do the two Arabic versions differ noticeably in which adjectives they use most? This could reflect structural differences between English and German that survive translation.
4. **Pokémon-specific vocabulary**: Are there adjectives that appear frequently in the game script but not in standard language corpora? (These might be interesting to flag.)

### Optional additions for later:
- **PyGal visualizations**: Bar charts of top 10 adjectives per language, or a grouped chart comparing adjective density across the four texts
- **WordNet lookup**: For the English adjectives, look up synset counts to measure semantic ambiguity (as in the original notebook)
- **Cross-language alignment**: For a deeper analysis, try to align sentences across the English and Arabic versions and see whether specific English adjectives consistently produce specific Arabic adjectives

## Acknowledgments

This notebook was developed with AI assistance from Anthropic's Claude Sonnet 4.6 
as part of the research workflow for this project. The pipeline design, 
code structure, and documentation were produced collaboratively between 
Dr. Elisa Beshero-Bondar and Claude over multiple sessions as a new learning resource for 
the Digital Media, Arts, and Technology program at Penn State Behrend. We sought 
assistance on Python approaches for natural language processing in Arabic 
and rendering right-to-left scripts. 

We include this acknowledgment as a model for transparent AI use in research and 
coursework. If you use or adapt this notebook, we encourage you to document AI 
assistance clearly, noting which tool you used, what you asked it to do, and how 
you verified or modified its output.

**Tools used in this project:**
- [spaCy](https://spacy.io/) — industrial-strength NLP for English and German POS tagging
- [CAMeL Tools](https://github.com/CAMeL-Lab/camel_tools) — Arabic NLP suite developed 
  by the CAMeL Lab at NYU Abu Dhabi
- [Claude](https://www.anthropic.com) (Anthropic) — AI assistant used in notebook development